# A HEP analysis with Dask (exploratory)

Adapted from [nsmith-/dask-hep-tutorial](https://github.com/nsmith-/dask-hep-tutorial/blob/main/03-hep.ipynb) — builds a dimuon invariant-mass analysis over real CMS Open Data using `dask`, `uproot`, `awkward`, and `hist`.

**This is exploratory, not a numbered lesson yet.** A few things are different from the rest of this training:

- **Python kernel, not bash.** Every other notebook here uses the bash kernel so each cell is an independent shell command. Dask needs a live `Client`/task graph that persists *across* cells in one Python process, so this notebook uses a real Python (`ipykernel`) kernel instead.
- **No grid certificate needed.** This uses CMS *Open Data* (`eospublic.cern.ch`), which is public — unlike the restricted CMS data in [CMS Data Access on NRP](../../lessons/5_cms_data.md), there's no X.509 proxy step here.
- **Runs entirely inside this one pod — no Kubernetes Job/Job manifests involved.** `distributed.Client()` below starts a *local* Dask cluster using this pod's own CPU/memory, the same way you'd use multiprocessing. Dask *can* scale out to a fleet of pods via `dask-kubernetes`'s `KubeCluster`, but that needs its own worker image and RBAC permissions to create pods — meaningfully more setup than this notebook needs. Start here; only reach for `KubeCluster` if a single pod's resources stop being enough.

The URLs and analysis steps below come directly from the source tutorial and haven't been independently re-verified from this environment — if a file path 404s, check the [source repo](https://github.com/nsmith-/dask-hep-tutorial) for an updated one.

## Install dependencies

Installs anything missing into your user site-packages. Reading `root://` URLs directly (no `xrdcp` download step) needs two separate packages: `xrootd` (the low-level Python bindings — a compiled package, so the one most likely to need attention if the hub image doesn't already have the underlying XRootD client libraries available) and `fsspec-xrootd` (registers `root://` as a protocol `fsspec` — and therefore uproot — knows how to open; without it you'll see `ModuleNotFoundError: No module named 'fsspec_xrootd'` the first time a cell tries to open a remote file).

In [ ]:
import importlib.util
import os
import subprocess
import sys

packages = [
    ("distributed", "dask[distributed]"),
    ("uproot", "uproot"),
    ("awkward", "awkward"),
    ("hist", "hist"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("vector", "vector"),
    ("pyarrow", "pyarrow"),
    ("XRootD", "xrootd"),
    ("fsspec_xrootd", "fsspec-xrootd"),
]
# cwd=~ avoids a pip bug where it crashes if the process's working directory
# no longer exists (os.getcwd() -> FileNotFoundError), which can happen on some
# JupyterHub setups.
home = os.path.expanduser("~")
for import_name, pip_name in packages:
    if importlib.util.find_spec(import_name) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--user", "--quiet", pip_name],
            check=True, cwd=home,
        )

# Some hub images (notably conda-based ones) don't put the --user
# site-packages directory on sys.path by default, so a successful pip
# install can still leave the import failing. Make sure it's there.
import site
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
print("Dependencies ready")


## Start a local Dask cluster

Sized conservatively on purpose — this pod likely shares a node with other users, and an unbounded `distributed.Client()` will try to use every core it can see. Adjust `n_workers`/`memory_limit` to match whatever this pod actually requests.

In [ ]:
import distributed

client = distributed.Client(n_workers=2, threads_per_worker=2, memory_limit="1.5GB")
client


**The dashboard link above is probably not clickable from here.** `distributed.Client()` prints a dashboard URL like `http://127.0.0.1:8787/status` — that's local to *this pod*, since the scheduler, workers, and dashboard all run in this same process. Opening it directly from your own browser won't work; you're not on the same network as the pod.

If this JupyterHub deployment has `jupyter-server-proxy` installed, the dashboard is usually reachable through the hub itself instead — try replacing the host/port with `<hub-url>/user/<your-username>/proxy/8787/status` (swap `8787` for whatever port the `client` repr above actually shows). If that 404s, this hub doesn't have that proxy set up, and the dashboard just isn't reachable from outside the pod here — that's a missing nice-to-have for watching task progress, not something the rest of this notebook depends on.

## Open the input files

CMS Open Data files from Run2012B/C double-muon datasets, ~2 GB each. Start with `N_FILES = 2` to keep a live demo quick — bump it up if it's running fast, drop to `1` if it's slow. We use `dask.delayed` to open each file's `Events` tree lazily, without reading any data yet.

In [ ]:
import dask
from dask import delayed
import uproot

N_FILES = 2

@delayed(pure=True)
def get_tree(url):
    return uproot.open(url)["Events"]

urls = [
    f"root://eospublic.cern.ch//eos/root-eos/benchmark/CMSOpenDataDimuon/Run2012BC_DoubleMuParked_Muons_{i}.root"
    for i in range(1, N_FILES + 1)
]
inputs = [get_tree(url) for url in urls]
inputs[0]


Compute one input and bring it back to inspect the available columns:

In [ ]:
inputs[0].compute().show()


## Build muon four-vectors

The original tutorial used `coffea.nanoevents.methods.vector`; we use the standalone [`vector`](https://vector.readthedocs.io/) package directly instead — same Lorentz-vector behavior, one fewer (and lighter) dependency.

In [ ]:
import awkward as ak
import vector

vector.register_awkward()

@delayed(pure=True)
def muon_struct(tree, entry_start, entry_stop):
    def get(name):
        return tree[name].array(entry_start=entry_start, entry_stop=entry_stop)

    return ak.zip(
        {
            "pt": get("Muon_pt"),
            "eta": get("Muon_eta"),
            "phi": get("Muon_phi"),
            "mass": get("Muon_mass"),
            "charge": get("Muon_charge"),
        },
        with_name="Momentum4D",
    )


## Chunk the files

Each file is a couple of GB, so we chunk by event range rather than reading a whole tree into memory at once. First, find out how many events are in each file (this triggers real reads, but only of metadata):

In [ ]:
nevents, = dask.compute([t.num_entries for t in inputs])
nevents


In [ ]:
import numpy as np

def chunks(n, target_size):
    edges = np.linspace(0, n, max(n // target_size, 1), dtype=int)
    return zip(edges[:-1], edges[1:])

chunksize = 400_000
muons = [
    muon_struct(tree, start, stop)
    for tree, nev in zip(inputs, nevents)
    for start, stop in chunks(nev, chunksize)
]
len(muons), muons[0]


[`persist`](https://docs.dask.org/en/latest/api.html#dask.persist) turns a lazy Dask object into one that's actively computed (or already computed) in the cluster, rather than only computed on demand. Persist the first chunk so later cells can reuse it quickly, and pull a copy back to the client for a quick look with matplotlib.

In [ ]:
muons[0] = muons[0].persist()
muons_chunk = muons[0].compute()


In [ ]:
import matplotlib.pyplot as plt

plt.hist(ak.num(muons_chunk), bins=range(6))
plt.xlabel("Muons per event")
plt.ylabel("Events")


In [ ]:
plt.hist(muons_chunk[ak.num(muons_chunk) == 2].sum().mass, bins=np.geomspace(0.2, 200, 200))
plt.gca().set_xscale("log")
plt.xlabel("Di-muon invariant mass [GeV]")
plt.ylabel("Events")


## Fill histograms across the cluster

The plots above pulled one chunk back to this notebook and used regular matplotlib. For the full dataset we instead want each chunk histogrammed on the worker that holds it, then the histograms summed — never pulling raw event data back to the client at all.

In [ ]:
import hist

@delayed(pure=True)
def nmuons_plot(muons_chunk):
    return (
        hist.Hist.new
        .Reg(6, 0, 6, name="nMuons")
        .Double()
        .fill(ak.num(muons_chunk))
    )

@delayed(pure=True)
def filter_muons(muons_chunk):
    return muons_chunk[
        (ak.num(muons_chunk) == 2)
        & (ak.sum(muons_chunk.charge, axis=1) == 0)
    ]

@delayed(pure=True)
def dimuon_cand(muons_chunk):
    return muons_chunk.sum()

@delayed(pure=True)
def mass_plot(cand_chunk):
    return (
        hist.Hist.new
        .Log(1000, 0.2, 200, name="mass", label="Di-muon invariant mass")
        .Double()
        .fill(cand_chunk.mass)
    )


`dask.compute` on several delayed objects at once lets the scheduler spot shared sub-computations (like `filter_muons` below being used by both a histogram and a table) and only run them once:

In [ ]:
test_nmu, test_mass = dask.compute(
    nmuons_plot(muons[0]),
    mass_plot(dimuon_cand(filter_muons(muons[0]))),
)
display(test_nmu, test_mass)


## Also keep a skimmed table

Alongside the histograms, save a reduced table of dimuon-candidate kinematics — one row per candidate event.

In [ ]:
import pandas as pd

@delayed(pure=True)
def cand_table(muons_chunk, cand_chunk):
    mupos = ak.firsts(muons_chunk[muons_chunk.charge == 1])
    muneg = ak.firsts(muons_chunk[muons_chunk.charge == -1])
    return pd.DataFrame({
        "mass": cand_chunk.mass,
        "pt": cand_chunk.pt,
        "mu+_pt": mupos.pt,
        "mu+_eta": mupos.eta,
        "mu+_phi": mupos.phi,
        "mu-_pt": muneg.pt,
        "mu-_eta": muneg.eta,
        "mu-_phi": muneg.phi,
    })

cand_table(
    filter_muons(muons[0]),
    dimuon_cand(filter_muons(muons[0])),
).compute()


## Visualize the task graph (optional)

`dask.visualize` needs the system `graphviz` binary (`dot`), not just the Python package — it may not be installed on the hub image. Wrapped so the rest of the notebook still works if it's missing.

In [ ]:
try:
    display(dask.visualize(
        (nmuons_plot(muons[0]), mass_plot(dimuon_cand(filter_muons(muons[0])))),
        optimize_graph=True,
    ))
except Exception as exc:
    print(f"Skipping graph visualization ({exc})")


## Reduce over all chunks and save results

Sum the per-chunk histograms with a Dask [bag](https://docs.dask.org/en/latest/bag-api.html) `fold`, and write the skimmed table (restricted to a Z-boson mass window, 60-120 GeV) to Parquet with a Dask [dataframe](https://docs.dask.org/en/latest/dataframe-api.html).

In [ ]:
import dask.bag as db
import dask.dataframe as dd
from operator import add

@delayed(pure=True)
def to_list(*args):
    return list(args)

def results(muons_chunk):
    filtered = filter_muons(muons_chunk)
    cand = dimuon_cand(filtered)
    return (
        nmuons_plot(muons_chunk),
        mass_plot(cand),
        cand_table(filtered, cand),
    )

result_set = np.array([results(chunk) for chunk in muons])
group = max(len(result_set) // 4, 1)
nmuons_final = db.from_delayed(
    to_list(*result_set[start:stop, 0])
    for start, stop in chunks(len(result_set), group)
).fold(add)
mass_final = db.from_delayed(
    to_list(*result_set[start:stop, 1])
    for start, stop in chunks(len(result_set), group)
).fold(add)
table_full = dd.from_delayed(result_set[:, 2])
table_skim = (
    table_full[abs(table_full.mass - 90.0) < 30.0]
    .repartition(2)
    .to_parquet("zmmtable", compute=False)
)


While this runs, check the Dask dashboard (the URL in the `client` repr a few cells up) to watch task progress across the cluster.

In [ ]:
res, = dask.compute((nmuons_final, mass_final, table_skim))


In [ ]:
fig, ax = plt.subplots()
res[0].plot(ax=ax)
ax.set_xlabel("Number of muons")
ax.set_ylabel("Event counts")


In [ ]:
fig, ax = plt.subplots()
res[1].plot(ax=ax)
ax.set_xscale("log")
ax.set_xlabel("Di-muon invariant mass [GeV]")
ax.set_ylabel("Event counts")


In [ ]:
pd.read_parquet("zmmtable")


## Clean up

Memory usage may look high even though the only thing persisted is `muons[0]` — workers hold onto memory the OS gave them until they're told to give it back, not because of a leak. Restarting clears it.

In [ ]:
client.restart()


In [ ]:
client.close()


## Where to go from here

If this is worth turning into a real numbered lesson: scale `N_FILES` and `n_workers` up, decide whether to formalize the CMS Open Data source (vs. reusing the restricted dataset from [CMS Data Access on NRP](../../lessons/5_cms_data.md)), and — only if a single pod's CPU/memory genuinely isn't enough — look at `dask-kubernetes`'s `KubeCluster` to run workers as separate pods instead of local threads.